Here is how you can load the datasets with S3
1. In Config file: LOAD_FROM_S3=True
2. Enter you SSP Cloud secrets in a .env file (see READme)

In [5]:
#Imports
import s3fs
import pandas as pd
import Config

from src.data_processing.data_load import data_loading
from src.data_processing.pipeline_processing import preprocessing_pipeline

In [ ]:
#To load data the from the different files directly
portefeuille, consommations, reclamations, interactions, impayes= data_loading("training")

/home/onyxia/work/Alptis-churn-prediction/src/data_processing/data_load.py:44: DtypeWarning: Columns (31,33) have mixed types. Specify dtype option on import or set low_memory=False.
  data_frames[key] = pd.read_csv(f, sep=";", index_col=False)


In [3]:
#Then to process the data (aggregate, merge, clean,...)
processed_dataset=preprocessing_pipeline(portefeuille, consommations, reclamations, interactions, impayes)

/home/onyxia/work/Alptis-churn-prediction/src/data_processing/cleaning.py:27: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_portefeuille_cleaned["client_departement"].replace("00", pd.NA, inplace=True)


/home/onyxia/work/Alptis-churn-prediction/src/data_processing/aggregation.py:117: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_agg["info_par_annee_mois_paiement"] = df.groupby("client_code").apply(build_info_dict).reset_index(drop=True)
/home/onyxia/work/Alptis-churn-prediction/src/data_processing/aggregation.py:161: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_mail = df[df["interaction_canal"] == "E-mail"].g

- The processing pipeline takes a lot of time (around 2 minutes)
- **Alternative option**: load the already merged dataset (in parquet format) stored in S3, in the group-2 bucket "projet-bdc-alptis-g2" (around 10 sec)

In [ ]:
#Use s3fs library to load the bucket where the parquet is stored
fs = s3fs.S3FileSystem(
        endpoint_url='https://' + Config.S3_ENDPOINT,
        key=Config.S3_ACCESS_KEY,
        secret=Config.S3_SECRET_KEY,
        token=Config.S3_SESSION_TOKEN,
        client_kwargs={'verify': Config.S3_VERIFY_SSL}
    )

#Load the merged file (stored in parquet format) in the bucket
with fs.open(Config.MERGED_PARQUET_S3, "rb") as f:
    df_merged= pd.read_parquet(f)

In [7]:
df_merged

,client_code,client_age_souscripteur,client_code_postal,client_departement,client_a_conjoint,client_nb_enfants,client_nb_assures_contrat,client_a_garantie_prevoyance,client_produit,client_garantie_base,...,interaction_service_cotisations_&_recouvrements,interaction_service_prestations_sante,interaction_service_suivi_du_contrat,interaction_autres_services,interaction_canal_telephone_nb,interaction_nb_transferts_reclamation,interaction_nb_transferts_gestion,interaction_nb_transferts_commercial,interaction_nb_transferts_externalises,interaction_historique_mail
0,21736,97,69100,69,0.0,0.0,1.0,0,produit_21,garantie_3,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None
1,22037,95,26230,26,0.0,0.0,1.0,1,produit_21,garantie_5,...,0.0,0.0,0.0,2.0,1.0,0.0,1.0,0.0,1.0,"{'2022-12-01': None, '2022-12-02': None, '2022..."
2,22331,88,69760,69,1.0,0.0,2.0,0,produit_21,garantie_3,...,0.0,10.0,0.0,0.0,10.0,0.0,5.0,0.0,7.0,None
3,22604,95,69003,69,0.0,0.0,1.0,0,produit_21,garantie_5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None
4,22702,90,69100,69,1.0,0.0,2.0,1,produit_21,garantie_5,...,0.0,8.0,0.0,0.0,8.0,0.0,4.0,0.0,8.0,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
132571,16204966,36,74140,74,0.0,0.0,1.0,0,produit_32,garantie_139,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,"{'2022-12-01': None, '2022-12-02': None, '2022..."
132572,16205008,63,17443,17,0.0,0.0,1.0,0,produit_20,garantie_100,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None
132573,16205281,25,74200,74,0.0,0.0,1.0,0,produit_32,garantie_137,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None
132574,16206464,22,97215,97,0.0,0.0,1.0,1,produit_18,garantie_87,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None
